In [4]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

# Load the breast cancer dataset
data = load_breast_cancer()

# Convert to a pandas DataFrame
df = pd.DataFrame(data.data, columns=data.feature_names)

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# Target variable
y = data.target

# Features
X = df

# 70%-30% train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Logistic regression model
logistic = LogisticRegression(
    max_iter=10000,
    solver="liblinear",
    random_state=42
)

# Recursive Feature Elimination: select exactly 2 features
rfe = RFE(
    estimator=logistic,
    n_features_to_select=2,
    step=1
)

# Fit RFE on training data only
rfe.fit(X_train, y_train)

# Get selected features
selected_features = X_train.columns[rfe.support_]

print("Selected features:")
print(selected_features.tolist())

# Ranking of all features
feature_ranking = pd.DataFrame({
    "Feature": X_train.columns,
    "Ranking": rfe.ranking_,
    "Selected": rfe.support_
}).sort_values("Ranking")

print("\nFeature ranking:")
print(feature_ranking)

# Use only the selected two features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# Scale features and train logistic regression
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=10000,
        solver="liblinear",
        random_state=42
    ))
])

model.fit(X_train_selected, y_train)

# Predictions
y_pred = model.predict(X_test_selected)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)

print("\nBest two features:")
for feature in selected_features:
    print("-", feature)

print(f"\nTest accuracy: {accuracy:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred))


Selected features:
['worst radius', 'worst concavity']

Feature ranking:
                    Feature  Ranking  Selected
26          worst concavity        1      True
20             worst radius        1      True
0               mean radius        2     False
6            mean concavity        3     False
27     worst concave points        4     False
7       mean concave points        5     False
22          worst perimeter        6     False
12          perimeter error        7     False
25        worst compactness        8     False
13               area error        9     False
11            texture error       10     False
28           worst symmetry       11     False
8             mean symmetry       12     False
9    mean fractal dimension       13     False
29  worst fractal dimension       14     False
4           mean smoothness       15     False
21            worst texture       16     False
5          mean compactness       17     False
16          concavity error       